# Dippy Animation Trajectory Generator

This notebook launches the Dippy app — a Gradio interface for chained WAN 2.1 Image-to-Video generation with frame continuity.

**Requirements:** A Colab runtime with GPU (T4 or better). An HF token and optionally an OpenAI API key configured in Colab Secrets.

In [4]:
# Cell 1: Mount Google Drive (for model caching)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!mkdir /root/.ssh/

# Copy '/content/drive/Colab Notebooks/utils/id_ed25519' to /root/.ssh
!cp '/content/drive/MyDrive/Colab Notebooks/utils/id_ed25519' /root/.ssh/.
!chmod 700 /root/.ssh
!chmod 600 /root/.ssh/id_ed25519
!ssh-keyscan -t ed25519 github.com >> /root/.ssh/known_hosts
!chmod 644 /root/.ssh/known_hosts
!ssh -T git@github.com || true

# github.com:22 SSH-2.0-b1ddf78
Hi dfu99! You've successfully authenticated, but GitHub does not provide shell access.


In [ ]:
# Cell 2: Clone the dippy-WAN repo
import os
if not os.path.exists('/content/dippy-WAN'):
    # Option A: Clone from GitHub (update URL to your repo)
    !git clone git@github.com:dfu99/dippy-WAN.git /content/dippy-WAN
else:
    print('dippy-WAN directory found.')


Cloning into '/content/dippy-WAN'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 29 (delta 10), reused 26 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 22.27 KiB | 11.13 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [ ]:
# Cell 3: Install latest-compatible dependencies (WAN + Transformers 5)
!pip install -q --upgrade \
  "diffusers==0.36.0" \
  "transformers==5.1.0" \
  "accelerate==1.12.0" \
  "huggingface_hub==1.4.1" \
  "gradio==6.5.1" \
  spaces ftfy peft imageio-ffmpeg opencv-python safetensors sentencepiece openai

# Print resolved versions so mismatches are obvious
import importlib.metadata as md
for pkg in ("diffusers", "transformers", "accelerate", "huggingface_hub", "gradio"):
    try:
        print(f"{pkg}=={md.version(pkg)}")
    except Exception:
        print(f"{pkg} not installed")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 144.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.7/374.7 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 115.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.2/105.2 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 123.7 MB/s eta 0:00:00
diffusers==0.34.0.dev0
transformers==4.55.4
accelerate==1.10.0
huggingf

In [5]:
# Cell 4: Configure cache, sync Drive -> local SSD, and set API keys
import os
from google.colab import userdata

DRIVE_CACHE_DIR = '/content/drive/My Drive/huggingface_cache_store'
LOCAL_CACHE_DIR = '/content/hf_cache'
# LOCAL_CACHE_DIR = DRIVE_CACHE_DIR
os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)

print('Syncing existing cache from Drive to local SSD...')
!rsync -a --ignore-existing --info=stats1 "{DRIVE_CACHE_DIR}/" "{LOCAL_CACHE_DIR}/"

print('Removing tiny/corrupt safetensors placeholders from local cache...')
!find "{LOCAL_CACHE_DIR}" -type f -name "*.safetensors" -size -16k -print -delete

# Use local SSD for runtime I/O reliability and speed
os.environ['HF_HOME'] = LOCAL_CACHE_DIR
os.environ['HF_HUB_CACHE'] = LOCAL_CACHE_DIR
os.environ['HF_ASSETS_CACHE'] = os.path.join(LOCAL_CACHE_DIR, 'assets')
os.environ['HF_XET_CACHE'] = os.path.join(LOCAL_CACHE_DIR, 'xet')
for key in ('HF_HOME', 'HF_HUB_CACHE', 'HF_ASSETS_CACHE', 'HF_XET_CACHE'):
    os.makedirs(os.environ[key], exist_ok=True)

# Keep Drive path so we can sync back later
os.environ['DIPPY_DRIVE_CACHE_DIR'] = DRIVE_CACHE_DIR

print(f"HF_HOME: {os.environ['HF_HOME']}")
print(f"HF_HUB_CACHE: {os.environ['HF_HUB_CACHE']}")
print(f"HF_ASSETS_CACHE: {os.environ['HF_ASSETS_CACHE']}")
print(f"HF_XET_CACHE: {os.environ['HF_XET_CACHE']}")
print(f"DIPPY_DRIVE_CACHE_DIR: {os.environ['DIPPY_DRIVE_CACHE_DIR']}")

# Hugging Face token (required for model download)
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded successfully.')
except Exception as e:
    print(f'Could not load HF_TOKEN: {e}')
    print('Please add a secret named HF_TOKEN in the Colab secrets tab (key icon).')

# OpenAI API key (optional, for LLM sentence generation)
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('OPENAI_API_KEY loaded successfully.')
except Exception as e:
    print(f'Could not load OPENAI_API_KEY: {e}')
    print('Sentence generation via LLM will not be available.')
    print('You can still type sentences manually.')


Syncing existing cache from Drive to local SSD...

sent 134,469,900,066 bytes  received 2,112 bytes  68,870,628.52 bytes/sec
total size is 134,437,067,101  speedup is 1.00
Removing tiny/corrupt safetensors placeholders from local cache...
HF_HOME: /content/hf_cache
HF_HUB_CACHE: /content/hf_cache
HF_ASSETS_CACHE: /content/hf_cache/assets
HF_XET_CACHE: /content/hf_cache/xet
DIPPY_DRIVE_CACHE_DIR: /content/drive/My Drive/huggingface_cache_store
HF_TOKEN loaded successfully.
OPENAI_API_KEY loaded successfully.


In [ ]:
# Cell 5: Launch the Dippy app
!python /content/dippy-WAN/dippy-app.py

2026-02-11 20:12:15.935680: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-11 20:12:15.955366: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770840735.978329   33729 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770840735.985983   33729 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770840736.005926   33729 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
# Cell 6: Sync local SSD cache back to Drive (run after successful model load)
import os

drive_cache = os.environ.get('DIPPY_DRIVE_CACHE_DIR', '/content/drive/My Drive/huggingface_cache_store')
local_cache = os.environ.get('HF_HUB_CACHE', '/content/hf_cache')
os.makedirs(drive_cache, exist_ok=True)

print(f'Syncing local cache to Drive: {local_cache} -> {drive_cache}')
print('Using rsync -L to copy real file contents (avoids broken symlink stubs).')
!rsync -aL --info=stats1 "{local_cache}/" "{drive_cache}/"
print('Drive sync complete.')
